# Ingestion Pipeline Test

Pipeline: **DOCX / PDF** → Normalize → Chunking → Embedding → ChromaDB

Xử lý tất cả dữ liệu trong `standalone/` và `pairs/` (trừ file changes.docx).  
Notebook là nơi thử nghiệm — tất cả hàm được định nghĩa trực tiếp, sau đó mới chuyển thành module `src/`.


In [1]:
# === Cell 1: Setup paths + imports cơ bản ===
import sys, json, re, os
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / "data" / "sample_pairs"
STANDALONE  = DATA_DIR / "standalone"
PAIRS_DIR   = DATA_DIR / "pairs"
CHROMA_DIR  = ROOT / "chroma_db"
OUTPUT_DIR  = ROOT / "outputs"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thu thập tất cả file DOCX/PDF (bỏ qua file "changes" và file tạm của Word)
all_files = []
for folder in [STANDALONE, PAIRS_DIR]:
    for f in sorted(folder.glob("*.docx")) + sorted(folder.glob("*.pdf")):
        name_lower = f.name.lower()
        if "changes" in name_lower or f.name.startswith("~$"):
            continue
        all_files.append(f)

print(f"ROOT: {ROOT}")
print(f"Tổng file cần xử lý: {len(all_files)}")
for f in all_files:
    print(f"  {f.relative_to(DATA_DIR)}")


ROOT: d:\Data Self Learning\Extra Projects\rag-based-legal-comparison
Tổng file cần xử lý: 10
  standalone\112_2025_QH15_586814.docx
  standalone\114_2025_QH15_658530.docx
  standalone\116_2025_QH15_666020.docx
  standalone\127_2025_QH15_686325.docx
  standalone\133_2025_QH15_675211.docx
  standalone\135_2025_QH15_675213.docx
  standalone\143_2025_QH15_681550.docx
  standalone\148_2025_QH15_675262.docx
  pairs\sample v1.docx
  pairs\sample v2.docx


In [2]:
# === Cell 2: Hàm đọc file DOCX / PDF + giữ cấu trúc paragraph ===
from collections import defaultdict
from docx import Document
import pdfplumber
import zipfile
import xml.etree.ElementTree as ET

W_URI = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
W_NS = {"w": W_URI}


def _w_attr(name: str) -> str:
    return f"{{{W_URI}}}{name}"


def _read_docx_xml(path: Path, inner_path: str) -> ET.Element | None:
    with zipfile.ZipFile(path) as zf:
        try:
            data = zf.read(inner_path)
        except KeyError:
            return None
    return ET.fromstring(data)


def _parse_docx_style_map(path: Path) -> dict[str, dict]:
    tree = _read_docx_xml(path, "word/styles.xml")
    if tree is None:
        return {}

    styles = {}
    for style in tree.findall(".//w:style", W_NS):
        style_id = style.get(_w_attr("styleId"))
        if not style_id:
            continue

        name_el = style.find("w:name", W_NS)
        based_on_el = style.find("w:basedOn", W_NS)
        num_pr = style.find("w:pPr/w:numPr", W_NS)

        num_id = None
        ilvl = None
        if num_pr is not None:
            num_id_el = num_pr.find("w:numId", W_NS)
            ilvl_el = num_pr.find("w:ilvl", W_NS)
            if num_id_el is not None:
                num_id = num_id_el.get(_w_attr("val"))
            if ilvl_el is not None:
                ilvl = ilvl_el.get(_w_attr("val"))

        styles[style_id] = {
            "style_id": style_id,
            "style_name": name_el.get(_w_attr("val")) if name_el is not None else style_id,
            "based_on": based_on_el.get(_w_attr("val")) if based_on_el is not None else None,
            "num_id": int(num_id) if num_id is not None else None,
            "ilvl": int(ilvl) if ilvl is not None else None,
        }

    return styles


def _resolve_style_numbering(style_id: str | None, style_map: dict[str, dict], seen: set[str] | None = None) -> tuple[int | None, int | None]:
    if not style_id or style_id not in style_map:
        return None, None

    seen = seen or set()
    if style_id in seen:
        return None, None
    seen.add(style_id)

    info = style_map[style_id]
    if info["num_id"] is not None:
        return info["num_id"], info["ilvl"] if info["ilvl"] is not None else 0

    return _resolve_style_numbering(info["based_on"], style_map, seen)


def _parse_docx_numbering(path: Path) -> dict:
    tree = _read_docx_xml(path, "word/numbering.xml")
    numbering = {"num_to_abstract": {}, "abstract_levels": {}}
    if tree is None:
        return numbering

    for abstract in tree.findall("w:abstractNum", W_NS):
        abstract_id = abstract.get(_w_attr("abstractNumId"))
        if abstract_id is None:
            continue

        levels = {}
        for lvl in abstract.findall("w:lvl", W_NS):
            ilvl_raw = lvl.get(_w_attr("ilvl"))
            if ilvl_raw is None:
                continue

            start_el = lvl.find("w:start", W_NS)
            fmt_el = lvl.find("w:numFmt", W_NS)
            text_el = lvl.find("w:lvlText", W_NS)
            p_style_el = lvl.find("w:pStyle", W_NS)
            levels[int(ilvl_raw)] = {
                "start": int(start_el.get(_w_attr("val"))) if start_el is not None else 1,
                "num_fmt": fmt_el.get(_w_attr("val")) if fmt_el is not None else "decimal",
                "lvl_text": text_el.get(_w_attr("val")) if text_el is not None else f"%{int(ilvl_raw) + 1}",
                "p_style": p_style_el.get(_w_attr("val")) if p_style_el is not None else None,
            }

        numbering["abstract_levels"][int(abstract_id)] = levels

    for num in tree.findall("w:num", W_NS):
        num_id = num.get(_w_attr("numId"))
        abstract_el = num.find("w:abstractNumId", W_NS)
        if num_id is None or abstract_el is None:
            continue
        numbering["num_to_abstract"][int(num_id)] = int(abstract_el.get(_w_attr("val")))

    return numbering


def _to_roman(value: int) -> str:
    roman_pairs = [
        (1000, "M"),
        (900, "CM"),
        (500, "D"),
        (400, "CD"),
        (100, "C"),
        (90, "XC"),
        (50, "L"),
        (40, "XL"),
        (10, "X"),
        (9, "IX"),
        (5, "V"),
        (4, "IV"),
        (1, "I"),
    ]
    result = []
    remaining = max(1, value)
    for arabic, roman in roman_pairs:
        while remaining >= arabic:
            result.append(roman)
            remaining -= arabic
    return "".join(result)


def _to_alpha(value: int, uppercase: bool = False) -> str:
    chars = []
    remaining = max(1, value)
    while remaining > 0:
        remaining -= 1
        chars.append(chr(ord("A" if uppercase else "a") + (remaining % 26)))
        remaining //= 26
    return "".join(reversed(chars))


def _format_counter(value: int, num_fmt: str) -> str:
    if num_fmt in {"upperRoman", "romanUpper"}:
        return _to_roman(value)
    if num_fmt in {"lowerRoman", "romanLower"}:
        return _to_roman(value).lower()
    if num_fmt == "upperLetter":
        return _to_alpha(value, uppercase=True)
    if num_fmt == "lowerLetter":
        return _to_alpha(value, uppercase=False)
    return str(value)


def _render_numbering_label(
    num_id: int,
    ilvl: int,
    numbering_data: dict,
    counters_by_num: defaultdict[int, dict[int, int]],
) -> tuple[str, list[str]]:
    abstract_id = numbering_data["num_to_abstract"].get(num_id)
    levels = numbering_data["abstract_levels"].get(abstract_id, {}) if abstract_id is not None else {}
    level_info = levels.get(ilvl, {})

    counters = counters_by_num[num_id]
    start = level_info.get("start", 1)
    counters[ilvl] = counters.get(ilvl, start - 1) + 1
    for level in list(counters):
        if level > ilvl:
            del counters[level]

    label = level_info.get("lvl_text", f"%{ilvl + 1}")
    path_parts = []
    for level in range(ilvl + 1):
        if level not in counters:
            continue
        num_fmt = levels.get(level, {}).get("num_fmt", "decimal")
        formatted = _format_counter(counters[level], num_fmt)
        label = label.replace(f"%{level + 1}", formatted)
        path_parts.append(formatted)

    label = re.sub(r"\s+", " ", label).strip()
    return label, path_parts


def _extract_paragraph_numbering(paragraph) -> tuple[int | None, int | None]:
    p_pr = paragraph._p.pPr
    num_pr = p_pr.numPr if p_pr is not None else None
    if num_pr is None:
        return None, None

    num_id_el = num_pr.numId
    ilvl_el = num_pr.ilvl
    num_id = int(num_id_el.val) if num_id_el is not None else None
    ilvl = int(ilvl_el.val) if ilvl_el is not None else 0
    return num_id, ilvl


def _get_heading_level(style_name: str | None, style_id: str | None) -> int | None:
    for candidate in (style_name, style_id):
        if not candidate:
            continue
        normalized = re.sub(r"\s+", "", candidate).lower()
        if normalized.startswith("heading") and normalized[7:].isdigit():
            return int(normalized[7:])
    return None


def read_docx_document(path: Path) -> dict:
    """Đọc DOCX theo paragraph và dựng lại nhãn numbering hiển thị khi có thể."""
    doc = Document(path)
    style_map = _parse_docx_style_map(path)
    numbering_data = _parse_docx_numbering(path)
    counters_by_num: defaultdict[int, dict[int, int]] = defaultdict(dict)
    paragraphs = []

    for idx, paragraph in enumerate(doc.paragraphs):
        text = paragraph.text.strip()
        if not text:
            continue

        style_name = paragraph.style.name if paragraph.style else None
        style_id = paragraph.style.style_id if paragraph.style else None
        heading_level = _get_heading_level(style_name, style_id)

        num_id, ilvl = _extract_paragraph_numbering(paragraph)
        if num_id is None:
            num_id, ilvl = _resolve_style_numbering(style_id, style_map)

        numbering_label = None
        numbering_path: list[str] = []
        if num_id is not None and ilvl is not None:
            numbering_label, numbering_path = _render_numbering_label(
                num_id=num_id,
                ilvl=ilvl,
                numbering_data=numbering_data,
                counters_by_num=counters_by_num,
            )

        display_text = text
        if heading_level in {1, 2, 3} and numbering_label:
            display_text = f"{numbering_label} {text}".strip()

        paragraphs.append(
            {
                "idx": idx,
                "text": text,
                "display_text": display_text,
                "style_name": style_name,
                "style_id": style_id,
                "heading_level": heading_level,
                "num_id": num_id,
                "ilvl": ilvl,
                "numbering_label": numbering_label,
                "numbering_path": numbering_path,
            }
        )

    return {
        "kind": "docx",
        "text": "\n".join(p["display_text"] for p in paragraphs),
        "paragraphs": paragraphs,
    }


def read_docx_text(path: Path) -> str:
    """Giữ API cũ: trả về plain text đã dựng lại heading nếu có numbering metadata."""
    return read_docx_document(path)["text"]


def read_pdf_text(path: Path) -> str:
    """Đọc file PDF text-based."""
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text.strip())
    return "\n".join(pages)


def read_document(path: Path) -> dict:
    """Đọc file DOCX/PDF và giữ cấu trúc paragraph khi khả dụng."""
    ext = path.suffix.lower()
    if ext == ".docx":
        return read_docx_document(path)
    if ext == ".pdf":
        text = read_pdf_text(path)
        paragraphs = [
            {
                "idx": idx,
                "text": line.strip(),
                "display_text": line.strip(),
                "style_name": None,
                "style_id": None,
                "heading_level": None,
                "num_id": None,
                "ilvl": None,
                "numbering_label": None,
                "numbering_path": [],
            }
            for idx, line in enumerate(text.splitlines())
            if line.strip()
        ]
        return {"kind": "pdf", "text": text, "paragraphs": paragraphs}
    raise ValueError(f"Không hỗ trợ định dạng: {ext}")


print("✅ Đã định nghĩa: read_document() dạng paragraph-aware, read_docx_document(), read_pdf_text()")

✅ Đã định nghĩa: read_document() dạng paragraph-aware, read_docx_document(), read_pdf_text()


In [3]:
# === Cell 3: Hàm normalize văn bản pháp luật ===

def normalize_text(text: str) -> str:
    """Chuẩn hoá text pháp luật: bỏ NBSP, chuẩn newline, gom khoảng trắng."""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_document(document: dict | str) -> dict | str:
    """Chuẩn hoá cả plain text lẫn document dạng paragraph records."""
    if isinstance(document, str):
        return normalize_text(document)

    paragraphs = []
    for para in document.get("paragraphs", []):
        text = normalize_text(para.get("text", ""))
        display_text = normalize_text(para.get("display_text") or text)
        if not display_text:
            continue

        normalized = dict(para)
        normalized["text"] = text
        normalized["display_text"] = display_text
        paragraphs.append(normalized)

    normalized_text = "\n".join(para["display_text"] for para in paragraphs)
    return {**document, "paragraphs": paragraphs, "text": normalized_text}


print("✅ Đã định nghĩa: normalize_text(), normalize_document()")

✅ Đã định nghĩa: normalize_text(), normalize_document()


In [4]:
# === Cell 4: Hàm chunking văn bản pháp luật / hợp đồng theo Điều ===

# --- Regex patterns cho cấu trúc luật Việt Nam ---
CHUONG_RE = re.compile(r"^Chương\s+([IVXLCDM\d]+)[\.:]?\s*(.*)", re.IGNORECASE)
MUC_RE = re.compile(r"^Mục\s+(\d+)[\.:]?\s*(.*)", re.IGNORECASE)
DIEU_RE = re.compile(r"^(?:Điều|ĐIỀU|dieu|DIEU)\s+((?:\d+[A-Za-z]?|[IVXLCDM]+))[\.:]?\s*(.*)$")
KHOAN_RE = re.compile(r"^(\d+)[\.)]\s*(.*)$")
DIEM_RE = re.compile(r"^([a-zđ])[\.)]\s*(.*)$", re.IGNORECASE)
DINH_NGHIA_RE = re.compile(r"giải thích từ ngữ|định nghĩa|diễn giải", re.IGNORECASE)


def _clean_heading(line: str) -> str:
    return re.sub(r"[:\s]+$", "", line.strip())


def _looks_upper_heading(line: str) -> bool:
    stripped = line.strip()
    if not stripped or stripped.endswith((".", ";", ",", ":")):
        return False
    if any(ch.isdigit() for ch in stripped):
        return False
    words = stripped.split()
    if len(words) < 2 or len(words) > 14:
        return False
    letters = [ch for ch in stripped if ch.isalpha()]
    if not letters:
        return False
    return sum(ch.isupper() for ch in letters) / len(letters) >= 0.8


def _looks_title_heading(line: str) -> bool:
    stripped = line.strip()
    if not stripped or stripped.endswith((".", ";", ",", ":")):
        return False
    if any(ch.isdigit() for ch in stripped):
        return False
    if any(ch in stripped for ch in ('"', "'", "(", ")", "-", "•", "*")):
        return False

    words = [re.sub(r"[^\wÀ-ỹĐđ-]", "", word) for word in stripped.split()]
    words = [word for word in words if word]
    if len(words) < 2 or len(words) > 14:
        return False

    capitalized = sum(1 for word in words if word[0].isupper())
    return capitalized >= max(2, len(words) - 1)


def _is_contract_section_heading(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if CHUONG_RE.match(stripped) or MUC_RE.match(stripped) or DIEU_RE.match(stripped):
        return False
    if KHOAN_RE.match(stripped) or DIEM_RE.match(stripped):
        return False
    if stripped.startswith(("(", "-", "•", "*", '"', "“")):
        return False
    if len(stripped) > 120:
        return False
    return _looks_upper_heading(stripped) or _looks_title_heading(stripped)


def _extract_khoan_items(body_lines: list[str]) -> list[dict]:
    khoan_items = []
    current = None

    for raw_line in body_lines:
        line = raw_line.strip()
        if not line:
            if current and current["text"]:
                current["text"] += "\n"
            continue

        khoan_match = KHOAN_RE.match(line)
        if khoan_match:
            if current:
                current["text"] = current["text"].strip()
                khoan_items.append(current)
            current = {
                "khoan_number": khoan_match.group(1),
                "text": khoan_match.group(2).strip(),
                "diem_items": [],
                "tieu_muc_items": [],
            }
            continue

        diem_match = DIEM_RE.match(line)
        if diem_match and current is not None:
            current["diem_items"].append(
                {
                    "diem_key": diem_match.group(1),
                    "text": diem_match.group(2).strip(),
                }
            )
            current["text"] = (current["text"] + "\n" + line).strip()
            continue

        if current is not None:
            current["text"] = (current["text"] + "\n" + line).strip()

    if current:
        current["text"] = current["text"].strip()
        khoan_items.append(current)

    return khoan_items


def _build_chunk_payload(
    *,
    chunk_idx: int,
    doc_id: str,
    version: str,
    chuong_so: str,
    muc_so: str,
    clause_id: str,
    article_number: str,
    article_title: str,
    text: str,
    khoan_items: list[dict] | None = None,
    chunk_type: str | None = None,
    created_at: str | None = None,
    tieu_muc_count: int | None = None,
    chunk_id_override: str | None = None,
    is_header: bool = False,
) -> dict:
    khoan_items = khoan_items or []
    diem_count = sum(len(item.get("diem_items", [])) for item in khoan_items)
    if tieu_muc_count is None:
        tieu_muc_count = sum(len(item.get("tieu_muc_items", [])) for item in khoan_items)

    payload = {
        "chunk_id": chunk_id_override or f"{doc_id}_{version}_{chunk_idx:04d}",
        "doc_id": doc_id,
        "version": version,
        "chunk_type": chunk_type or ("definition" if DINH_NGHIA_RE.search(article_title) else "article"),
        "chuong_so": chuong_so,
        "muc_so": muc_so,
        "clause_id": clause_id,
        "article_number": article_number,
        "article_title": article_title,
        "khoan_count": 0 if is_header else len(khoan_items),
        "diem_count": 0 if is_header else diem_count,
        "tieu_muc_count": 0 if is_header else tieu_muc_count,
        "khoan_items": [] if is_header else khoan_items,
        "text": text,
        "char_len": len(text),
        "created_at": created_at or datetime.now(timezone.utc).isoformat(),
    }
    return payload


def _chunk_docx_headings(document: dict, doc_id: str, version: str) -> list[dict]:
    paragraphs = document.get("paragraphs", [])
    chunks = []
    chunk_idx = 1

    article_state = None
    current_khoan = None
    current_tieu_muc = None

    def flush_article() -> None:
        nonlocal article_state, current_khoan, current_tieu_muc, chunk_idx
        if article_state is None:
            return

        block_lines = [article_state["heading_display"]] + article_state["body_lines"]
        block = "\n".join(line for line in block_lines if line.strip()).strip()
        if block:
            chunks.append(
                _build_chunk_payload(
                    chunk_idx=chunk_idx,
                    doc_id=doc_id,
                    version=version,
                    chuong_so="0",
                    muc_so="0",
                    clause_id=f"article_{article_state['article_number']}",
                    article_number=article_state["article_number"],
                    article_title=article_state["article_title"],
                    text=block,
                    khoan_items=article_state["khoan_items"],
                )
            )
            chunk_idx += 1

        article_state = None
        current_khoan = None
        current_tieu_muc = None

    for para in paragraphs:
        text = para.get("text", "").strip()
        display_text = para.get("display_text") or text
        heading_level = para.get("heading_level")
        numbering_label = para.get("numbering_label")
        numbering_path = para.get("numbering_path") or []

        if heading_level == 1 and numbering_label:
            flush_article()
            article_number = numbering_path[0] if numbering_path else numbering_label.rstrip(".")
            article_state = {
                "article_number": article_number,
                "article_title": _clean_heading(text),
                "heading_display": display_text.strip(),
                "body_lines": [],
                "khoan_items": [],
            }
            continue

        if article_state is None:
            continue

        if heading_level == 2 and numbering_label:
            current_khoan = {
                "khoan_number": numbering_label.rstrip("."),
                "title": _clean_heading(text),
                "text": display_text.strip(),
                "diem_items": [],
                "tieu_muc_items": [],
            }
            current_tieu_muc = None
            article_state["khoan_items"].append(current_khoan)
            article_state["body_lines"].append(display_text.strip())
            continue

        if heading_level == 3 and numbering_label:
            if current_khoan is None:
                synthetic_number = ".".join(numbering_path[:2]) if len(numbering_path) >= 2 else numbering_label.rstrip(".")
                current_khoan = {
                    "khoan_number": synthetic_number,
                    "title": "",
                    "text": "",
                    "diem_items": [],
                    "tieu_muc_items": [],
                }
                article_state["khoan_items"].append(current_khoan)

            current_tieu_muc = {
                "tieu_muc_number": numbering_label.rstrip("."),
                "title": _clean_heading(text),
                "text": display_text.strip(),
            }
            current_khoan["tieu_muc_items"].append(current_tieu_muc)
            current_khoan["text"] = "\n".join(part for part in [current_khoan["text"], display_text.strip()] if part).strip()
            article_state["body_lines"].append(display_text.strip())
            continue

        article_state["body_lines"].append(display_text.strip())
        if current_tieu_muc is not None:
            current_tieu_muc["text"] = (current_tieu_muc["text"] + "\n" + display_text.strip()).strip()
            current_khoan["text"] = (current_khoan["text"] + "\n" + display_text.strip()).strip()
        elif current_khoan is not None:
            current_khoan["text"] = (current_khoan["text"] + "\n" + display_text.strip()).strip()

    flush_article()
    return chunks


def _chunk_plain_text_hierarchy(text: str, doc_id: str, version: str) -> list[dict]:
    """Chunk theo Điều; nếu văn bản không có Điều thì fallback theo tiêu đề section."""
    lines = [line.rstrip() for line in text.split("\n")]
    lines = [line for line in lines if line.strip()]
    chunks = []
    chunk_idx = 1
    section_seq = 1

    cur_chuong_so = "0"
    cur_muc_so = "0"

    explicit_article_count = sum(1 for line in lines if DIEU_RE.match(line.strip()))
    use_explicit_articles = explicit_article_count > 0

    current_article_no = None
    current_clause_id = None
    current_heading_lines = []
    current_title_lines = []
    current_body_lines = []
    preamble_finished = False

    def start_section(heading_line: str, article_number: str | None = None, title_text: str | None = None) -> None:
        nonlocal current_article_no, current_clause_id, current_heading_lines, current_title_lines, current_body_lines, section_seq

        if article_number is None:
            current_article_no = str(section_seq)
            current_clause_id = f"section_{section_seq:03d}"
            section_seq += 1
        else:
            current_article_no = str(article_number)
            current_clause_id = f"article_{article_number}"

        clean_title = _clean_heading(title_text or heading_line)
        current_heading_lines = [heading_line.strip()]
        current_title_lines = [clean_title] if clean_title else []
        current_body_lines = []

    def flush_section() -> None:
        nonlocal chunk_idx, current_article_no, current_clause_id, current_heading_lines, current_title_lines, current_body_lines

        if current_article_no is None:
            return

        if current_clause_id.startswith("section_") and not any(line.strip() for line in current_body_lines):
            current_article_no = None
            current_clause_id = None
            current_heading_lines = []
            current_title_lines = []
            current_body_lines = []
            return

        block_lines = current_heading_lines + current_body_lines
        block = "\n".join(line for line in block_lines if line.strip()).strip()
        if not block:
            current_article_no = None
            current_clause_id = None
            current_heading_lines = []
            current_title_lines = []
            current_body_lines = []
            return

        article_title = " - ".join(line for line in current_title_lines if line.strip())
        if not article_title:
            article_title = " - ".join(_clean_heading(line) for line in current_heading_lines)

        khoan_items = _extract_khoan_items(current_body_lines)
        chunks.append(
            _build_chunk_payload(
                chunk_idx=chunk_idx,
                doc_id=doc_id,
                version=version,
                chuong_so=cur_chuong_so,
                muc_so=cur_muc_so,
                clause_id=current_clause_id,
                article_number=current_article_no,
                article_title=article_title,
                text=block,
                khoan_items=khoan_items,
            )
        )
        chunk_idx += 1
        current_article_no = None
        current_clause_id = None
        current_heading_lines = []
        current_title_lines = []
        current_body_lines = []

    for line in lines:
        stripped = line.strip()

        match_chuong = CHUONG_RE.match(stripped)
        match_muc = MUC_RE.match(stripped)
        match_dieu = DIEU_RE.match(stripped)

        if match_chuong:
            flush_section()
            cur_chuong_so = match_chuong.group(1)
            cur_muc_so = "0"
            preamble_finished = True
            continue

        if match_muc:
            flush_section()
            cur_muc_so = match_muc.group(1)
            preamble_finished = True
            continue

        if match_dieu:
            flush_section()
            article_number = match_dieu.group(1)
            article_title = match_dieu.group(2).strip() or f"Điều {article_number}"
            start_section(stripped, article_number=article_number, title_text=article_title)
            preamble_finished = True
            continue

        if use_explicit_articles:
            if current_article_no is not None:
                current_body_lines.append(stripped)
            continue

        if _is_contract_section_heading(stripped) and preamble_finished:
            if current_article_no is None:
                start_section(stripped)
            elif current_body_lines:
                flush_section()
                start_section(stripped)
            else:
                current_heading_lines.append(stripped)
                current_title_lines.append(_clean_heading(stripped))
            continue

        if current_article_no is not None:
            current_body_lines.append(stripped)

    flush_section()
    return chunks


def chunk_full_hierarchy(document: dict | str, doc_id: str, version: str) -> list[dict]:
    """Chunk theo Điều; ưu tiên paragraph-aware DOCX nếu có Heading/numbering metadata."""
    if isinstance(document, dict):
        paragraphs = document.get("paragraphs", [])
        numbered_heading_count = sum(
            1
            for para in paragraphs
            if para.get("heading_level") == 1 and para.get("numbering_label")
        )
        if numbered_heading_count > 0:
            return _chunk_docx_headings(document, doc_id=doc_id, version=version)
        text = document.get("text", "")
        return _chunk_plain_text_hierarchy(text, doc_id=doc_id, version=version)

    return _chunk_plain_text_hierarchy(document, doc_id=doc_id, version=version)


print("✅ Đã định nghĩa lại chunker: bỏ header, bỏ chuong_ten/muc_ten, giữ article/definition có giá trị")

✅ Đã định nghĩa lại chunker: bỏ header, bỏ chuong_ten/muc_ten, giữ article/definition có giá trị


In [9]:
# === Cell 5: Đọc + Normalize + Chunk tất cả file ===

all_chunks: dict[str, list] = {}  # key = "doc_id__version"

for file_path in all_files:
    doc_id = file_path.stem
    # Xác định version từ tên file
    name_lower = file_path.name.lower()
    if "v1" in name_lower:
        version = "v1"
    elif "v2" in name_lower:
        version = "v2"
    else:
        version = "v1"  # standalone mặc định v1

    raw = read_document(file_path)
    clean = normalize_document(raw)
    chunks = chunk_full_hierarchy(clean, doc_id=doc_id, version=version)

    key = f"{doc_id}__{version}"
    all_chunks[key] = chunks

    n_article = sum(1 for c in chunks if c["chunk_type"] == "article")
    n_def = sum(1 for c in chunks if c["chunk_type"] == "definition")
    print(f"✅ {file_path.name:45s} → {len(chunks):3d} chunks (article={n_article}, definition={n_def})")

total = sum(len(v) for v in all_chunks.values())
print(f"\nTổng: {total} chunks từ {len(all_chunks)} bộ dữ liệu")

✅ 112_2025_QH15_586814.docx                     →  58 chunks (article=57, definition=1)
✅ 114_2025_QH15_658530.docx                     →  46 chunks (article=45, definition=1)
✅ 116_2025_QH15_666020.docx                     →  45 chunks (article=44, definition=1)
✅ 127_2025_QH15_686325.docx                     → 180 chunks (article=179, definition=1)
✅ 133_2025_QH15_675211.docx                     →  27 chunks (article=26, definition=1)
✅ 135_2025_QH15_675213.docx                     →  95 chunks (article=94, definition=1)
✅ 143_2025_QH15_681550.docx                     →  52 chunks (article=51, definition=1)
✅ 148_2025_QH15_675262.docx                     →  48 chunks (article=47, definition=1)
✅ sample v1.docx                                →  21 chunks (article=21, definition=0)
✅ sample v2.docx                                →  21 chunks (article=21, definition=0)

Tổng: 593 chunks từ 10 bộ dữ liệu


In [6]:
# === Cell 6: Kiểm thử chất lượng chunking ===

errors = []

for key, chunks in all_chunks.items():
    doc_id = key.split("__")[0]

    if not chunks:
        errors.append(f"{doc_id}: không có chunk nào")
        continue

    ids = [c["chunk_id"] for c in chunks]
    if len(ids) != len(set(ids)):
        errors.append(f"{doc_id}: chunk_id bị trùng")

    empty = [c for c in chunks if c["char_len"] <= 30]
    if empty:
        errors.append(f"{doc_id}: {len(empty)} chunk quá ngắn: {[c['clause_id'] for c in empty]}")

    if not all(c["chunk_type"] in ("article", "definition") for c in chunks):
        errors.append(f"{doc_id}: chunk_type không hợp lệ")

    chuongs = sorted(set(c["chuong_so"] for c in chunks if c["chuong_so"] != "0"))
    print(f"  {doc_id:45s} | {len(chunks):3d} điều khoản | Chương: {chuongs or 'N/A'}")

if errors:
    print(f"\n❌ {len(errors)} lỗi:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"\n✅ Tất cả {len(all_chunks)} bộ dữ liệu PASS kiểm thử chunking")


  112_2025_QH15_586814                          |  58 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI']
  114_2025_QH15_658530                          |  46 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI']
  116_2025_QH15_666020                          |  45 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']
  127_2025_QH15_686325                          | 180 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'IX', 'V', 'VI', 'VII', 'VIII', 'X', 'XI', 'XII', 'XIII', 'XIV', 'XV']
  133_2025_QH15_675211                          |  27 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI']
  135_2025_QH15_675213                          |  95 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']
  143_2025_QH15_681550                          |  52 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII']
  148_2025_QH15_675262                          |  48 điều khoản | Chương: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']


In [7]:
# === Cell 7: Xem mẫu chunk (definition / article) ===

sample_key = list(all_chunks.keys())[0]
sample_chunks = all_chunks[sample_key]
print(f"Mẫu từ: {sample_key}\n")

for ctype in ["definition", "article"]:
    found = next((c for c in sample_chunks if c["chunk_type"] == ctype), None)
    if found:
        print(f"{'─'*65}")
        print(f"  chunk_type    : {found['chunk_type']}")
        print(f"  clause_id     : {found['clause_id']}")
        print(f"  article_title : {found['article_title'] or '(không có)'}")
        print(f"  chuong_so     : {found['chuong_so']}  |  muc_so: {found['muc_so']}")
        print(f"  khoan_count   : {found['khoan_count']}  |  diem_count: {found['diem_count']}")
        print(f"  text ({found['char_len']} ký tự):")
        print(f"    {found['text'][:200].strip()}")
        print()


Mẫu từ: 112_2025_QH15_586814__v1

─────────────────────────────────────────────────────────────────
  chunk_type    : definition
  clause_id     : article_3
  article_title : Giải thích từ ngữ
  chuong_so     : I  |  muc_so: 0
  khoan_count   : 22  |  diem_count: 0
  text (6280 ký tự):
    Điều 3. Giải thích từ ngữ
Trong Luật này, các từ ngữ dưới đây được hiểu như sau:
1. Quy hoạch là định hướng phát triển, sắp xếp, phân bố không gian các hoạt động kinh tế, xã hội, quốc phòng, an ninh g

─────────────────────────────────────────────────────────────────
  chunk_type    : article
  clause_id     : article_1
  article_title : Phạm vi điều chỉnh
  chuong_so     : I  |  muc_so: 0
  khoan_count   : 0  |  diem_count: 0
  text (227 ký tự):
    Điều 1. Phạm vi điều chỉnh
Luật này quy định hệ thống quy hoạch; việc lập, thẩm định, quyết định hoặc phê duyệt, công bố, cung cấp thông tin, thực hiện, đánh giá và điều chỉnh quy hoạch; quản lý nhà n



In [11]:
# === Cell 8: Export chunks ra JSONL + JSON (khong dung chunk_type/clause_id) ===

exported = []

for key, chunks in all_chunks.items():
    jsonl_path = OUTPUT_DIR / f"{key}_chunks.jsonl"
    json_path = OUTPUT_DIR / f"{key}_chunks.json"

    cleaned_chunks = []
    for c in chunks:
        c2 = dict(c)
        c2.pop("chunk_type", None)
        c2.pop("clause_id", None)
        cleaned_chunks.append(c2)

    with jsonl_path.open("w", encoding="utf-8") as f:
        for c in cleaned_chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")

    json_path.write_text(
        json.dumps(cleaned_chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    exported.append((jsonl_path.name, json_path.name, len(cleaned_chunks)))
    print(f"  Saved: {jsonl_path.name}  ({len(cleaned_chunks)} chunks)")
    print(f"  Saved: {json_path.name}  ({len(cleaned_chunks)} chunks)")

print()
for jsonl_name, json_name, expected in exported:
    jsonl_file = OUTPUT_DIR / jsonl_name
    json_file = OUTPUT_DIR / json_name

    loaded_jsonl = [json.loads(line) for line in jsonl_file.read_text(encoding="utf-8").strip().split("\n") if line.strip()]
    loaded_json = json.loads(json_file.read_text(encoding="utf-8"))

    assert len(loaded_jsonl) == expected, f"[FAIL] {jsonl_name}: {len(loaded_jsonl)} != {expected}"
    assert len(loaded_json) == expected, f"[FAIL] {json_name}: {len(loaded_json)} != {expected}"

    assert all("chunk_type" not in row for row in loaded_jsonl), f"[FAIL] {jsonl_name}: van con chunk_type"
    assert all("chunk_type" not in row for row in loaded_json), f"[FAIL] {json_name}: van con chunk_type"
    assert all("clause_id" not in row for row in loaded_jsonl), f"[FAIL] {jsonl_name}: van con clause_id"
    assert all("clause_id" not in row for row in loaded_json), f"[FAIL] {json_name}: van con clause_id"

    print(f"  [OK] {jsonl_name}: {len(loaded_jsonl)} chunks doc lai OK")
    print(f"  [OK] {json_name}: {len(loaded_json)} chunks doc lai OK")

print(f"\n[OK] Export JSONL + JSON: {len(exported)} bo file -> outputs/ (khong chunk_type, khong clause_id)")

  Saved: 112_2025_QH15_586814__v1_chunks.jsonl  (58 chunks)
  Saved: 112_2025_QH15_586814__v1_chunks.json  (58 chunks)
  Saved: 114_2025_QH15_658530__v1_chunks.jsonl  (46 chunks)
  Saved: 114_2025_QH15_658530__v1_chunks.json  (46 chunks)
  Saved: 116_2025_QH15_666020__v1_chunks.jsonl  (45 chunks)
  Saved: 116_2025_QH15_666020__v1_chunks.json  (45 chunks)
  Saved: 127_2025_QH15_686325__v1_chunks.jsonl  (180 chunks)
  Saved: 127_2025_QH15_686325__v1_chunks.json  (180 chunks)
  Saved: 133_2025_QH15_675211__v1_chunks.jsonl  (27 chunks)
  Saved: 133_2025_QH15_675211__v1_chunks.json  (27 chunks)
  Saved: 135_2025_QH15_675213__v1_chunks.jsonl  (95 chunks)
  Saved: 135_2025_QH15_675213__v1_chunks.json  (95 chunks)
  Saved: 143_2025_QH15_681550__v1_chunks.jsonl  (52 chunks)
  Saved: 143_2025_QH15_681550__v1_chunks.json  (52 chunks)
  Saved: 148_2025_QH15_675262__v1_chunks.jsonl  (48 chunks)
  Saved: 148_2025_QH15_675262__v1_chunks.json  (48 chunks)
  Saved: sample v1__v1_chunks.jsonl  (21 chunk

## Embedding + ChromaDB

Cells phía dưới cần GPU/model nặng. Chạy riêng khi cần index vào ChromaDB.


In [9]:
# === Cell 9: Hàm embedding + ChromaDB + Load model ===
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv(ROOT / ".env")
hf_token = os.getenv("HF_TOKEN")

EMBED_MODEL = "huyydangg/DEk21_hcmute_embedding"
COLLECTION_NAME = "legal_chunks"


def load_embedder() -> SentenceTransformer:
    """Load SentenceTransformer embedding model."""
    return SentenceTransformer(EMBED_MODEL, token=hf_token)


def get_collection(chroma_dir: Path, collection_name: str = COLLECTION_NAME):
    """Lấy hoặc tạo ChromaDB collection."""
    client = chromadb.PersistentClient(path=str(chroma_dir))
    return client.get_or_create_collection(collection_name)


embedder = load_embedder()
print(f"✅ Embedding model loaded: {embedder.get_sentence_embedding_dimension()} dim")


d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 400.26it/s, Materializing param=pooler.dense.weight]                               


✅ Embedding model loaded: 768 dim


In [ ]:
# === Cell 10: Hàm index chunks + Index tất cả vào ChromaDB (khong dung chunk_type/clause_id) ===
from pyvi import ViTokenizer


def _chunk_to_metadata(chunk: dict) -> dict:
    return {
        "doc_id": chunk["doc_id"],
        "version": chunk["version"],
        "chuong_so": chunk["chuong_so"],
        "muc_so": chunk["muc_so"],
        "article_number": chunk["article_number"],
        "article_title": chunk["article_title"],
        "khoan_count": chunk["khoan_count"],
        "diem_count": chunk["diem_count"],
        "tieu_muc_count": chunk.get("tieu_muc_count", 0),
    }


def index_chunks(chunks: list[dict], chroma_dir: Path, embedder: SentenceTransformer,
                 collection_name: str = "legal_chunks", batch_size: int = 64) -> int:
    """Embed chunks và upsert vào ChromaDB. Trả về số chunks đã index."""
    if not chunks:
        return 0

    collection = get_collection(chroma_dir, collection_name)

    texts = [c["text"] for c in chunks]
    segmented = [ViTokenizer.tokenize(t) for t in texts]
    vectors = embedder.encode(segmented, convert_to_numpy=True, show_progress_bar=True)

    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        batch = chunks[start:end]
        collection.upsert(
            ids=[c["chunk_id"] for c in batch],
            documents=[c["text"] for c in batch],
            metadatas=[_chunk_to_metadata(c) for c in batch],
            embeddings=[v.tolist() for v in vectors[start:end]],
        )
    return len(chunks)


# --- Index tất cả ---
total_indexed = 0
for key, chunks in all_chunks.items():
    cleaned = [dict(c) for c in chunks]
    for c in cleaned:
        c.pop("chunk_type", None)
        c.pop("clause_id", None)
    n = index_chunks(cleaned, chroma_dir=CHROMA_DIR, embedder=embedder)
    total_indexed += n
    print(f"  Indexed: {key} -> {n} chunks")

print(f"\n[OK] Tong: {total_indexed} chunks da index vao ChromaDB")

d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\.venv\Lib\site-packages\pyvi\ViTokenizer.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  model = pickle.load(fin)
Batches: 100%|██████████| 3/3 [00:02<00:00,  1.28it/s]


  Indexed: 112_2025_QH15_586814__v1 → 65 chunks


Batches: 100%|██████████| 2/2 [00:00<00:00,  4.98it/s]


  Indexed: 114_2025_QH15_658530__v1 → 52 chunks


Batches: 100%|██████████| 2/2 [00:00<00:00,  4.15it/s]


...

  Indexed: 148_2025_QH15_675262__v1 → 56 chunks


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]


  Indexed: sample v1__v1 → 29 chunks


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]

  Indexed: sample v2__v2 → 29 chunks

✅ Tổng: 678 chunks đã index vào ChromaDB


In [ ]:
# === Cell 11: Kiểm tra ChromaDB (khong dung chunk_type/clause_id) ===

collection = get_collection(CHROMA_DIR)
total = collection.count()
print(f"Tong chunks trong ChromaDB: {total}")

all_meta = collection.get(include=["metadatas"])["metadatas"]
doc_counts = Counter(m["doc_id"] for m in all_meta)

print("\n-- So chunks theo doc_id --")
for doc, cnt in sorted(doc_counts.items()):
    print(f"  {doc}: {cnt}")

required = ["doc_id", "version", "article_number", "article_title"]
missing = [m for m in all_meta if not all(f in m for f in required)]
if missing:
    print(f"\n[WARN] {len(missing)} chunk thieu metadata!")
else:
    print(f"\n[OK] Tat ca {total} chunks co du metadata")

Tổng chunks trong ChromaDB: 678

── Số chunks theo doc_id ──
  112_2025_QH15_586814: 65
  114_2025_QH15_658530: 52
  116_2025_QH15_666020: 53
  127_2025_QH15_686325: 194
  133_2025_QH15_675211: 32
  135_2025_QH15_675213: 103
  143_2025_QH15_681550: 65
  148_2025_QH15_675262: 56
  sample v1: 29
  sample v2: 29

── Số chunks theo chunk_type ──
  article: 658
  definition: 10
  header: 10

✅ Tất cả 678 chunks có đủ metadata


In [ ]:
# === Cell 12: Xem mẫu từng loại chunk trong ChromaDB ===

for chunk_type in ["definition", "article"]:
    result = collection.get(
        where={"chunk_type": chunk_type},
        limit=1,
        include=["metadatas", "documents", "embeddings"],
    )
    if not result["ids"]:
        print(f"Không có chunk loại '{chunk_type}'")
        continue

    meta = result["metadatas"][0]
    text = result["documents"][0]
    emb  = result["embeddings"][0]

    print(f"{'─'*65}")
    print(f"  chunk_type    : {meta['chunk_type']}")
    print(f"  chunk_id      : {result['ids'][0]}")
    print(f"  doc_id        : {meta['doc_id']}")
    print(f"  clause_id     : {meta['clause_id']}")
    print(f"  article_title : {meta['article_title'] or '(không có)'}")
    print(f"  text ({len(text)} ký tự):")
    print(f"    {text[:200].strip()}")
    print(f"  embedding     : dim={len(emb)}, [{emb[0]:.4f}, {emb[1]:.4f}, ... {emb[-1]:.4f}]")
    print()


─────────────────────────────────────────────────────────────────
  chunk_type    : header
  chunk_id      : 112_2025_QH15_586814_v1_0000
  doc_id        : 112_2025_QH15_586814
  clause_id     : header
  article_title : (không có)
  text (191 ký tự):
    LUẬT
QUY HOẠCH
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Quy hoạch.
QUY ĐỊNH CHUNG
  embedding     : dim=768, [-0.1595, -0.0582, ... 0.5254]

─────────────────────────────────────────────────────────────────
  chunk_type    : definition
  chunk_id      : 112_2025_QH15_586814_v1_0003
  doc_id        : 112_2025_QH15_586814
  clause_id     : article_3
  article_title : Điều 3. Giải thích từ ngữ
  text (6280 ký tự):
    Điều 3. Giải thích từ ngữ
Trong Luật này, các từ ngữ dưới đây được hiểu như sau:
1. Quy hoạch là định hướng phát triển, sắp xếp, phân bố không gian các hoạt động kinh tế, xã hội, quốc phòng, an ninh g
  embedding    